# Chapter 4: Foundational R for Precision Health

## 1. Introduction

Welcome to a pivotal step in your R programming journey. In previous sections, you learned to manipulate data manually. Now, you will learn to automate those manipulations. This section introduces custom functions and the `apply` family—a powerful set of tools for performing repetitive tasks across your data without writing cumbersome loops. We will start by creating a simple function to calculate drug dosage. Then, we will explore how `apply`, `lapply`, `sapply`, and `tapply` can be used to efficiently summarize entire datasets, such as calculating mean lab values for cohorts of patients or processing lists of clinical measurements. By the end, you will be able to write your own functions and choose the correct `apply` tool to streamline your precision health data analysis workflows.

---

## 2. Key Concepts and Definitions

*   **Function**: A reusable block of code that performs a specific task. In medicine, a function could be designed to calculate a patient's risk score based on multiple inputs (e.g., age, blood pressure, cholesterol), turning a complex, repetitive calculation into a single, reliable command.
*   **Default Parameter**: A value assigned to a function argument in its definition. If the user doesn't provide a value for that argument when calling the function, R automatically uses the default. This is useful for medical calculations where a standard value (e.g., a standard dose per kg) is common.
*   **Anonymous Function**: A function defined on-the-fly without being assigned a name. This is useful for simple, one-time operations, such as quickly converting a list of patient temperatures from Celsius to Fahrenheit directly within an `lapply` call.
*   **`apply()`**: A function used to apply another function to the rows or columns of a matrix. A clinical researcher might use `apply()` on a matrix of gene expression data to calculate the mean expression level for each gene (across patient samples) or for each patient (across all genes).
*   **`lapply()`**: An "list apply" function that applies a function to each element of a list and is guaranteed to return a list. This is ideal for processing nested patient data, where each element of a main list is itself a list containing a patient's demographics, lab results, and medications. `lapply` ensures the output structure remains a consistent list.
*   **`sapply()`**: A "simplifying apply" function that works like `lapply` but attempts to simplify the resulting list into a more user-friendly format, such as a vector or matrix. For example, if you calculate the Body Mass Index (BMI) for a list of patients, `sapply` would return the results as a simple, named vector of BMIs.
*   **`tapply()`**: A "tagged apply" function that splits a vector into groups based on a factor and then applies a function to each group. This is the cornerstone of subgroup analysis, such as calculating the mean drug response time (`vector`) for patients in different treatment arms (`factor`) of a clinical trial.

---

## 3. Main Content

### 3.1 Creating a Basic Function

You can define a custom function using the `function()` constructor to automate any repeated task. This promotes code that is readable, reusable, and less prone to errors. A common use case in a clinical setting is creating a function to calculate drug dosage based on patient weight.

> **Key Terms:** A **default parameter** is a value assigned to a function argument in its definition. If the user doesn't provide a value for that argument when calling the function, R automatically uses the default.

```R
# Define a function to calculate dosage from weight.
calculate_dosage <- function(weight_kg, dose_per_kg = 5) {
  # The last expression is automatically returned.
  weight_kg * dose_per_kg
}

# Calculate dosage for a 70kg patient at 10mg/kg.
calculate_dosage(weight_kg = 70, dose_per_kg = 10)
# [1] 700

# This call uses the default dose of 5mg/kg.
calculate_dosage(weight_kg = 70)
# [1] 350
```

### 3.2 The `apply()` Function for Matrices

`apply()` executes a function over the rows (`MARGIN=1`) or columns (`MARGIN=2`) of a matrix. It is ideal for summarizing matrix-like data, such as a table of lab results where rows are patients and columns are time points.

> **Important:** `apply()` converts data frames into matrices before operating. This process, known as type coercion, forces all columns to a single data type (e.g., numeric to character). If your data frame has mixed types, numeric functions like `mean()` will fail. Use `apply()` on matrices or data frames with all-numeric data.

> **Medical Background:** White Blood Cell (WBC) counts are a fundamental diagnostic tool. Analyzing trends over several days (rows) for a single patient helps monitor infection or treatment response, while analyzing counts across all patients for a single day (columns) could help spot a ward-wide issue or a lab equipment error.

```R
# Matrix of WBC counts (x10^9/L). Normal range is approx. 4.5-11.0.
wbc_counts <- matrix(c(8.1, 7.5, 9.2, 11.0,
                       6.5, 6.8, NA, 7.3,
                       12.1, 11.5, 11.9, 12.4),
                     nrow = 3, byrow = TRUE,
                     dimnames = list(c("PT001", "PT002", "PT003"),
                                     c("day_1", "day_2", "day_3", "day_4")))

# Calculate the mean WBC count for each patient (by row).
mean_per_patient <- apply(X = wbc_counts, MARGIN = 1, FUN = mean, na.rm = TRUE)
mean_per_patient
#    PT001    PT002    PT003 
# 8.950000 6.866667 11.975000
```

### 3.3 The `lapply()` Function for Lists

`lapply()` applies a function to each element of a list or vector and always returns a list. This is the safest and most predictable choice when your input is a list and you want to ensure the output is also a list, preserving the structure.

> **Key Terms:** An **anonymous function** is a function created on-the-fly without being assigned a name. It's useful for simple, one-time operations within functions like `lapply()` or `sapply()`, avoiding the need to define a separate named function.

```R
# List of patient body temperatures in Celsius.
patient_temps_c <- list(PT004 = 37.5, PT005 = 38.1, PT006 = 36.9)

# Convert Celsius to Fahrenheit using an anonymous function.
temps_f_list <- lapply(patient_temps_c, function(temp_c) (temp_c * 9/5) + 32)
temps_f_list
# $PT004
# [1] 99.5
# 
# $PT005
# [1] 100.58
# 
# $PT006
# [1] 98.42
```

### 3.4 The `sapply()` Function for Simplified Output

`sapply()` is a user-friendly version of `lapply()`. It applies a function to each list element but then attempts to simplify the result into a vector or matrix. For example, we can calculate the Body Mass Index (BMI) for a list of patients.

> **Important:** `sapply()` simplifies its output, but the result's format can be unpredictable (it might return a vector, matrix, or list depending on the input). For stable, predictable code where you always expect a list, it is safer to use `lapply()`.

```R
# List of patient metrics where each element is a named vector.
patient_metrics <- list(
  PT101 = c(weight_kg = 78, height_m = 1.75),
  PT102 = c(weight_kg = 92, height_m = 1.80)
)

# Use sapply to calculate BMI for each patient.
bmi_values <- sapply(patient_metrics, function(p) {
  p[["weight_kg"]] / (p[["height_m"]]^2)
})
bmi_values
#    PT101    PT102 
# 25.46939 28.39506
```

> **Pro Tip:** When extracting elements from a list or vector inside a function, using double square brackets `[[...]]` is often safer than a single bracket `[...]`. It ensures you are extracting a single, "un-boxed" element, which prevents unexpected errors in calculations.

### 3.5 The `tapply()` Function for Grouped Operations

`tapply()` is designed for "ragged" data, where you want to apply a function to subsets of a vector based on a grouping variable (a factor). This is extremely useful for comparing metrics across different cohorts.

> **In Practice:** Group-wise analysis with `tapply()` is a cornerstone of clinical data analysis. It allows you to quickly compare outcomes, vitals, or lab results between different patient groups, such as "treatment" vs. "control," or, as in this example, "Fever" vs. "Stable."

```R
# Vector of temperatures and a corresponding factor for patient condition.
temps <- c(37.1, 38.1, 36.9, 39.0)
condition <- factor(c("Stable", "Fever", "Stable", "Fever"))

# Calculate mean temperature for each condition.
mean_temp_by_condition <- tapply(X = temps, INDEX = condition, FUN = mean)
mean_temp_by_condition
#  Fever Stable 
#  38.55  37.00
```

---

## 4. Practice Exercises

### Exercise 1: Create and Apply a BMI Function

**Objective:** Write a custom function and apply it to a list of patient data.
**Time:** 10 minutes
**Medical Context:** Body Mass Index (BMI) is a standard screening metric for weight categories that may lead to health problems. Automating its calculation is a common task.

**Task:**
1.  Write a function named `calculate_bmi` that computes BMI.
2.  The function must accept a named vector containing `weight_kg` and `height_m`. The formula is `weight_kg / height_m^2`.
3.  Use `sapply()` to apply your function to the `patient_metrics` list below to get a named vector of all patient BMIs.

```R
# Practice dataset
patient_metrics <- list(
  PT101 = c(weight_kg = 78, height_m = 1.75),
  PT102 = c(weight_kg = 92, height_m = 1.80),
  PT103 = c(weight_kg = 65, height_m = 1.65)
)
```

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```R
# 1. Define the function.
calculate_bmi <- function(metrics) {
  # Use [[...]] to safely extract single named elements.
  metrics[["weight_kg"]] / (metrics[["height_m"]]^2)
}

# 2. Apply the function using sapply().
# We use sapply() because our function returns a single numeric value for each patient.
# This allows sapply() to neatly simplify the output into a named vector, which is easy to read and use.
bmi_results <- sapply(patient_metrics, calculate_bmi)
print(bmi_results)

# Expected Output:
#    PT101    PT102    PT103 
# 25.46939 28.39506 23.87512
```

**Explanation:** The `calculate_bmi` function correctly extracts weight and height from the input vector to compute the BMI. `sapply` iterates through each patient in the `patient_metrics` list, applies the function, and simplifies the list of single-number results into a convenient named vector.
**Key Learning:** Combining custom functions with `sapply` is a powerful pattern for processing structured lists.

> **Reflection Moment:** In this exercise, `sapply()` returned a simple vector. What do you think would happen if your `calculate_bmi` function returned two values instead of one (e.g., the BMI and a classification like "Normal")? Would `sapply()` still be the best choice?

</div>
</details>

### Exercise 2: Find Peak WBC Count Day

**Objective:** Use `apply()` with a custom function to find the column name (day) corresponding to the maximum value in each row.
**Time:** 15 minutes
**Medical Context:** Identifying the specific day a patient's white blood cell (WBC) count peaked can be crucial for correlating with infection onset or treatment response.

**Task:**
Using the `wbc_counts` matrix from the lesson, find the day (`day_1`, `day_2`, etc.) on which each patient had their highest WBC count. The `which.max()` function will be helpful, as it returns the *index* of the maximum value in a vector.

```R
# wbc_counts matrix from the lesson
wbc_counts <- matrix(c(8.1, 7.5, 9.2, 11.0,
                       6.5, 6.8, NA, 7.3,
                       12.1, 11.5, 11.9, 12.4),
                     nrow = 3, byrow = TRUE,
                     dimnames = list(c("PT001", "PT002", "PT003"),
                                     c("day_1", "day_2", "day_3", "day_4")))
```

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```R
peak_day <- apply(wbc_counts, 1, function(patient_row) {
  # Get the column names of the matrix
  days <- colnames(wbc_counts)
  # Find the index of the highest WBC count for the row, ignoring NA
  max_index <- which.max(patient_row)
  # Return the day name from that index
  return(days[max_index])
})
print(peak_day)

# Expected Output:
#   PT001   PT002   PT003 
# "day_4" "day_4" "day_4"
```

**Explanation:** We use `apply` to iterate over each row (`MARGIN = 1`). For each `patient_row`, our anonymous function finds the index of the maximum value using `which.max()`. It then uses this index to look up the corresponding column name from `colnames(wbc_counts)`.
**Key Learning:** `apply()` can be used with custom functions that do more than just summarize, such as finding the position of a key event.

</div>
</details>

### Exercise 3: Summarize Blood Pressure by Risk Group

**Objective:** Use `tapply()` to calculate multiple summary statistics for different patient groups.
**Time:** 10 minutes
**Medical Context:** Stratifying patients by risk group (e.g., Low, Medium, High) and analyzing their vitals is fundamental to population health management and identifying trends.

**Task:**
You are given a vector of systolic blood pressure readings and a corresponding factor indicating the risk group for each patient. Use `tapply()` to calculate the **mean** and **standard deviation** for each risk group.

```R
# Practice dataset
systolic_bp <- c(120, 135, 122, 145, 155, 138, 118, 160)
risk_group <- factor(c("Low", "Medium", "Low", "High", "High", "Medium", "Low", "High"))
```

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```R
# Calculate mean blood pressure by risk group
mean_bp_by_risk <- tapply(systolic_bp, risk_group, mean)
print(mean_bp_by_risk)

# Calculate standard deviation of blood pressure by risk group
sd_bp_by_risk <- tapply(systolic_bp, risk_group, sd)
print(sd_bp_by_risk)

# Expected Output:
#     High      Low   Medium 
# 153.3333 120.0000 136.5000 

#     High      Low   Medium 
# 7.637626 2.000000 2.121320
```

**Explanation:** `tapply` first groups the `systolic_bp` values according to the `risk_group` factor. The first call then computes the `mean()` for each of these groups (High, Low, Medium). The second call similarly computes the standard deviation (`sd()`) for the same groups.
**Key Learning:** `tapply` is the ideal tool for performing summary calculations on different subsets of your data, forming the basis of stratified analysis.

</div>
</details>

---

## 5. Practical Applications

*   **High-Throughput Genomic Data Analysis**: In genomics, researchers often work with large matrices where rows represent genes and columns represent patient samples. Using `apply(gene_expression_matrix, 1, mean)` allows for the rapid calculation of the average expression for every single gene across all patients, helping to identify which genes are most or least active in a cohort. This avoids slow and error-prone `for` loops.

*   **Processing Electronic Health Records (EHRs)**: EHR data for a patient cohort is often delivered as a list of lists, where each top-level element represents a patient and contains nested lists or vectors for demographics, lab results, and prescriptions. `lapply()` is perfect for this structure. A researcher could use `lapply(patient_list, extract_medications)` to run a custom function that pulls the medication history for every patient, returning a clean list of medication lists for further analysis.

*   **Clinical Trial Subgroup Analysis**: `tapply()` is the workhorse for analyzing clinical trial outcomes. Given a vector of patient outcomes (e.g., change in tumor size) and a factor indicating the treatment arm (e.g., "Placebo", "Drug A", "Drug B"), a biostatistician can use `tapply(outcome_vector, treatment_arm, mean)` to instantly get the average outcome for each group. This is a fundamental step in determining if a new therapy is effective.

---

## 6. Summary and Key Takeaways

In this section, we've explored how to make your R code more powerful and efficient by writing custom functions and using the `apply` family to automate repetitive analyses. We moved from manual calculations to scalable operations, a critical skill for handling the large and complex datasets found in precision health.

Key takeaways from this section include:

*   **Write Functions for Reusability**: Encapsulate any repeated logic into a custom function to make your code cleaner, easier to debug, and more reliable.
*   **Choose the Right `apply` Tool for the Job**:
    *   To operate on **rows/columns of a matrix**: Use `apply()`.
    *   To operate on **elements of a list** and get a **list back**: Use `lapply()`.
    *   To operate on **elements of a list** and get a **simplified vector/matrix**: Use `sapply()`.
    *   To operate on a **vector grouped by a factor**: Use `tapply()`.
*   **Vectorization over Loops**: The `apply` family provides a "vectorized" approach that is more idiomatic and computationally faster in R than writing your own `for` loops for most data summary tasks.

Mastering these functions will significantly accelerate your ability to explore and summarize clinical and biological data. In the next section, we will learn how to visualize these summaries effectively using `ggplot2`.

---